# 06 · Qué mira una CNN
*Filtros, activaciones, saliencia, oclusión y la imagen que más activa un canal*

Notebook del episodio 6 de **Visión: de convoluciones a ViT** ([ver la lección](https://d32vh5r3pvn9et.cloudfront.net/cursos/vision/que-mira-una-cnn)). Es el mismo guion del video, en texto, con cada bloque de código para correr y su salida real.

**Mecanismo:** conv1 de cnn_ep05.pt (32, 3, 5, 5), 74,7 % test: separar luz (promedio RGB) y color → 14 de 32 filtros más de color que de luz → hooks en las 3 ReLU con un avión negro sobre fondo blanco: activo 34 % / 20 % / 14 % → saliencia M = max_c |∂S_k/∂x| (Simonyan 2013): correlación entrenada vs. al azar 0,19 en 100 imágenes; el avión ocupa 11 % y recibe 23 % de la saliencia (al azar 15 %) → Adebayo 2018: control de red al azar → oclusión con parche gris 8×8: tapar el avión baja p 0,23, tapar fondo 0,24, la mayor caída (0,98) en el fondo → ascenso por gradiente sobre x en canales de conv3: texturas, no objetos

**Cómo correrlo:** con el entorno de `~/Movies/serie-vision/lab` (`cd lab && uv run --project ~/Movies/serie-alineacion/lab --group notebooks --with jupyterlab jupyter lab ../notebooks`, o en VS Code eligiendo el kernel `lab/.venv`). Todo corre offline: Gemma 2 2B sale de la caché de Hugging Face, y los entrenamientos son de juguete, con datos sintéticos.

Cada bloque de código arranca de cero (`%reset -f`): la celda de **preparación** es lo que el video no muestra, y la siguiente es exactamente el código que aparece en pantalla.

## ▶ Antes de empezar

Corré la celda de abajo una vez: baja el código de la serie y sus dependencias. Funciona en Google Colab (gratis) y en cualquier Jupyter. Para guardar tus cambios en Colab: **Archivo → Guardar una copia en Drive**.

In [ ]:
import os, subprocess, sys
SERIE = "vision"
REPO = "https://github.com/Faridmurzone/paperclase-notebooks"

def _raiz():  # carpeta del repo si ya estamos adentro
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, ".paperclase-notebooks")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

RAIZ = _raiz()
if RAIZ is None:
    RAIZ = os.path.abspath("paperclase-notebooks")
    if not os.path.isdir(RAIZ):
        subprocess.run(["git", "clone", "--depth", "1", REPO, RAIZ], check=True)
os.chdir(os.path.join(RAIZ, SERIE, "lab"))  # los bloques de código importan desde snippets/ y leen datos/
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)
os.environ["HF_HUB_OFFLINE"] = "0"  # el notebook original corría sin red; acá baja los modelos de Hugging Face
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("listo:", os.getcwd())

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import warnings; warnings.filterwarnings("ignore")

*Visión, de las convoluciones a los transformers. Episodio seis: qué mira una CNN, convolutional neural network. En el episodio cinco entrenamos una red convolucional chica que acierta tres de cada cuatro imágenes de test. Hoy la abrimos: miramos sus pesos, lo que calcula por dentro, y qué píxeles parecen importarle.*

### Cuatro formas de mirar adentro

- Los pesos: los filtros de la primera capa
- Las activaciones: qué canales se prenden y dónde
- La saliencia: el gradiente de la clase respecto de cada píxel
- Intervenir: tapar pedazos, o fabricar la imagen que más activa

Hay cuatro maneras de mirar una red por dentro. Mirar los pesos directamente: los de la primera capa se pueden dibujar como imágenes chiquitas. Mirar las activaciones, o sea, qué sale de cada capa para una imagen concreta, con hooks. Calcular la saliencia: cuánto cambiaría la clase si movieras un poco cada píxel, que es un gradiente. E intervenir: tapar pedazos de la imagen y ver qué pasa, o partir de ruido y modificarlo hasta que un canal se prenda al máximo. Las tres primeras observan; la cuarta cambia algo y mide.

## Parte 1 · Los filtros de la primera capa

La primera capa es la única que mira píxeles directamente. Sus pesos se pueden ver como imágenes.

### Los pesos de conv1

Cargamos la red del episodio cinco, ya entrenada. Los pesos de la primera convolución son un tensor de treinta y dos por tres por cinco por cinco: treinta y dos filtros, cada uno con un kernel de cinco por cinco para cada canal de color. Separamos cada filtro en dos partes: el promedio de los tres canales, que reacciona a la luz sin importar el color, y lo que queda, que es lo que distingue rojo de verde de azul. Contamos cuántos filtros tienen más de color que de luz. La red acierta setenta y cuatro coma siete por ciento en test, y catorce de los treinta y dos filtros son más de color que de luz.

In [2]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep06_filtros.py
__file__ = 'snippets/ep06_filtros.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, sys
import torch
AQUI = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, AQUI)
from _redes import cargar_modelo
from _figs import norm, mosaico, guardar
torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [3]:
cnn, info = cargar_modelo(f"{AQUI}/../datos/cnn_ep05.pt")   # la del ep05
W = cnn.conv1.weight                         # (32, 3, 5, 5)
print("conv1:", tuple(W.shape), " precisión test:", f"{info['precision_test']:.1%}")

gris_ = W.mean(dim=1, keepdim=True)          # lo común a R, G y B: luz
color = W - gris_                            # lo que cambia entre canales
r = color.flatten(1).norm(dim=1) / (gris_.flatten(1).norm(dim=1) * 3 ** 0.5)
print("filtros más de color que de luz:", (r > 1).sum().item(), "de 32")

conv1: (32, 3, 5, 5)  precisión test: 74.7%
filtros más de color que de luz: 14 de 32


In [4]:
# Verificación extra (corrió junto con el video)
orden = r.argsort()                          # primero los de luz
lum = gris_[:, 0]
guardar("ep06_filtros.json", {"filas": mosaico([lum[i] / lum.abs().max() for i in orden], 8),
                              "orden": orden.tolist()})

### Un filtro es una plantilla que se compara

$$
y_{u,v} = \textstyle\sum_{c}\sum_{i,j} W_{c,i,j}\; x_{c,\,u+i,\,v+j} + b
$$
<small>multiplicar el filtro por un pedazo de la imagen y sumar</small>

$$
\bar W_{i,j} = \tfrac{1}{3}\,(W_{R,i,j} + W_{G,i,j} + W_{B,i,j})
$$
<small>la parte de luz: igual en los tres canales</small>

$$
r = \frac{\lVert W - \bar W \rVert}{\sqrt{3}\,\lVert \bar W \rVert} > 1 \;\Rightarrow\; \text{filtro de color}
$$
<small>más diferencia entre canales que señal común</small>

La convolución, en cada posición u v, multiplica el filtro W por el pedazo de imagen que tiene abajo y suma todo: setenta y cinco productos, tres canales por cinco por cinco. En el dibujo, seis de esos productos y su suma. La suma es grande cuando el pedazo se parece al filtro: el filtro es una plantilla. La parte de luz, W barra, es el promedio de los tres canales. En el dibujo, rojo, verde y azul, cada uno por un tercio. Y r compara lo que cambia entre canales con esa parte común; la raíz de tres es porque W barra se repite en los tres canales. Si r pasa de uno, decimos que el filtro es de color.

### Los 32 filtros de conv1 (parte de luz)

Acá están los treinta y dos, cada uno es un cuadrado de cinco por cinco. Celeste es peso positivo y rojo es negativo. Los de arriba a la izquierda son los más de luz: varios tienen una franja positiva al lado de una negativa, que es un detector de borde, como el kernel de Sobel del episodio dos, pero aprendido solo. Los últimos, los de color, se ven apagados en esta vista, porque su parte de luz es chica: lo que miran es la diferencia entre canales, y eso no se ve en un dibujo de un solo color.

## Parte 2 · Activaciones

Los pesos son lo mismo para todas las imágenes. Las activaciones cambian con cada una.

### Guardar lo que sale de cada capa

Usamos un avión negro sobre fondo blanco, del test. Un hook es una función que PyTorch llama cada vez que un módulo termina su forward; la nuestra guarda la salida en un diccionario. La colgamos después de cada una de las tres ReLU, rectified linear unit, la función que deja pasar lo positivo y pone cero en lo negativo. Una sola pasada hacia adelante, y los tres mapas quedan guardados. Medimos qué fracción de cada mapa es distinta de cero. Predice avión, y acierta. La primera capa da treinta y dos canales de treinta y dos por treinta y dos, con treinta y cuatro por ciento de activaciones prendidas; la segunda, veinte por ciento; la tercera, catorce.

In [5]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep06_activaciones.py
__file__ = 'snippets/ep06_activaciones.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, sys
import torch
AQUI = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, AQUI)
from _redes import cargar_modelo
from _cifar import cargar, normalizar, CLASES
from _figs import norm, gris, mosaico, guardar
torch.set_grad_enabled(False)
cnn, _ = cargar_modelo(f"{AQUI}/../datos/cnn_ep05.pt")
X, y = cargar("test", n=100)
i = 21                                             # un avión negro sobre fondo blanco
x = normalizar(X[i:i + 1])

In [6]:
mapas = {}
def guardar_salida(nombre):
    def hook(mod, ent, sal):
        mapas[nombre] = sal[0]                # (canales, alto, ancho)
    return hook

for k, nombre in [(1, "relu1"), (4, "relu2"), (7, "relu3")]:
    cnn.features[k].register_forward_hook(guardar_salida(nombre))
logits = cnn(x)
print("predice:", CLASES[logits.argmax().item()], "| es:", CLASES[y[i].item()])
for nombre, m in mapas.items():
    activos = (m > 0).float().mean().item()
    print(f"{nombre}: {tuple(m.shape)}  activo: {activos:.0%}")

predice: avión | es: avión
relu1: (32, 32, 32)  activo: 34%
relu2: (64, 16, 16)  activo: 20%
relu3: (64, 8, 8)  activo: 14%


In [7]:
# Verificación extra (corrió junto con el video)
m1 = mapas["relu1"]
top = m1.mean(dim=(1, 2)).argsort(descending=True)[:3]
paneles = [gris(X[i].float() / 255)] + [norm(m1[c]) for c in top]
guardar("ep06_activ.json", {"paneles": [p.tolist() for p in paneles], "canales": top.tolist(),
                            "relu2": [norm(mapas["relu2"][c]).tolist() for c in mapas["relu2"].mean(dim=(1, 2)).argsort(descending=True)[:3]]})

### El avión y tres canales de la primera capa

Arriba a la izquierda, el avión en escala de grises, con sus treinta y dos por treinta y dos píxeles. En los otros tres cuadros, los tres canales de la primera capa que más se prenden con esta imagen. Los tres se prenden a medias en el fondo claro, se apagan sobre el avión negro, y brillan fuerte en un borde distinto cada uno: el de atrás, el diagonal de abajo, el de la derecha. También se ve una línea en el borde de la imagen: es el padding de ceros. La primera capa todavía mira cosas locales y simples.

### Fracción de activaciones distintas de cero, por capa

La misma medición, como barras. En la primera capa, un tercio de las activaciones son positivas. En la segunda, una de cada cinco. En la tercera, una de cada siete. Las capas más profundas tienen campo receptivo más grande, como vimos en el episodio cuatro, y cada canal se prende solo cuando aparece su patrón, que ya es más específico que un borde. Ojo: es una sola imagen.

- **relu1 · 32×32**: 34 %
- **relu2 · 16×16**: 20 %
- **relu3 · 8×8**: 14 %

## Parte 3 · Saliencia

¿Qué píxeles, si los movieras un poquito, cambiarían más la respuesta?

### La saliencia es un gradiente

$$
S_k(x) \approx w^\top x + b,\quad w = \frac{\partial S_k}{\partial x}
$$
<small>cerca de x, la red se parece a una función lineal</small>

$$
M_{i,j} = \max_{c}\, \Big|\frac{\partial S_k}{\partial x_{c,i,j}}\Big|
$$
<small>un número por píxel: el canal de color con más efecto</small>

S sub k es el logit de la clase k, un número. Cerca de la imagen x, se aproxima por w transpuesta x más b, donde w es la derivada de S sub k respecto de cada píxel: un vector del mismo tamaño que la imagen, tres por treinta y dos por treinta y dos. En el dibujo, una neurona con tres píxeles: el que tiene derivada uno coma cinco pesa mucho más que los otros. La saliencia M es, para cada píxel, el valor absoluto de esa derivada, quedándose con el canal de color donde es más grande. En el dibujo, la derivada es la pendiente de la tangente: cuánto sube S si movés un poco x.

### Saliencia, con un control

Backward desde el logit de la clase correcta hasta la imagen. Valor absoluto y máximo sobre los tres canales. El control: la misma arquitectura sin entrenar, con pesos al azar. Si la saliencia de las dos se parece, el mapa no está diciendo nada de lo que la red aprendió. Y para el avión, qué parte de la saliencia cae sobre sus píxeles oscuros. En cien imágenes, la correlación entre entrenada y al azar es cero coma diecinueve: bastante distintas, pasa el control. El avión ocupa once por ciento de la imagen; la saliencia de la red entrenada pone veintitrés por ciento ahí, y la red al azar, quince.

In [8]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep06_saliencia.py
__file__ = 'snippets/ep06_saliencia.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, sys
import torch
AQUI = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, AQUI)
from _redes import cargar_modelo, CNNChica
from _cifar import cargar, normalizar
from _figs import norm, gris, mosaico, guardar
cnn, _ = cargar_modelo(f"{AQUI}/../datos/cnn_ep05.pt")
X, y = cargar("test", n=100)
torch.manual_seed(0)
al_azar = CNNChica().eval()                        # misma arquitectura, sin entrenar
i = 21                                             # un avión negro sobre fondo blanco


def corr(a, b):                                    # correlación de Pearson entre dos mapas
    a, b = a.flatten() - a.mean(), b.flatten() - b.mean()
    return (a @ b / (a.norm() * b.norm())).item()


def bordes(img):                                   # |diferencia con el vecino|: un detector de bordes sin red
    g = gris(img.float() / 255)
    return torch.nn.functional.pad((g[1:, :-1] - g[:-1, :-1]).abs() + (g[:-1, 1:] - g[:-1, :-1]).abs(), (0, 1, 0, 1))

In [9]:
def saliencia(red, x, clase):
    x = x.clone().requires_grad_(True)
    red(x)[0, clase].backward()               # ∂ logit / ∂ píxel
    return x.grad[0].abs().amax(dim=0)        # (32, 32): máx. sobre RGB

parecido = []
for j in range(100):                          # 100 imágenes de test
    x, k = normalizar(X[j:j + 1]), y[j].item()
    parecido.append(corr(saliencia(cnn, x, k), saliencia(al_azar, x, k)))
print("entrenada vs. al azar (100 imágenes):", round(sum(parecido) / 100, 2))

x, k = normalizar(X[i:i + 1]), y[i].item()  # el avión negro
avion = gris(X[i].float() / 255) < 0.5       # sus píxeles oscuros
print(f"el avión ocupa {avion.float().mean():.0%} de la imagen")
for nombre, red in [("entrenada", cnn), ("al azar", al_azar)]:
    s = saliencia(red, x, k)
    print(f"saliencia {nombre} sobre el avión: {s[avion].sum() / s.sum():.0%}")

entrenada vs. al azar (100 imágenes): 0.19
el avión ocupa 11% de la imagen
saliencia entrenada sobre el avión: 23%
saliencia al azar sobre el avión: 15%


In [10]:
# Verificación extra (corrió junto con el video)
guardar("ep06_sal.json", {"paneles": [gris(X[i].float() / 255).tolist(),
                                      norm(saliencia(cnn, x, k)).tolist(), norm(saliencia(al_azar, x, k)).tolist()]})

### Saliencia: red entrenada y red al azar

Las tres imágenes lado a lado. La saliencia de la red entrenada se junta alrededor del avión, aunque con mucho ruido de píxel a píxel; la de la red al azar está repartida por todos lados. Parece una explicación: la red mira el avión. Guardá esa impresión, porque en dos escenas la vamos a poner a prueba.

> **Simonyan, Vedaldi y Zisserman, 2013 — Deep Inside Convolutional Networks** · [arXiv:1312.6034](https://arxiv.org/abs/1312.6034)
>
> **M = max_c |∂S/∂x|** · Saliencia por clase «computing the gradient of the class score with respect to the input image»; el mapa toma «the maximum magnitude of w across all colour channels».

Esta es la receta original. Simonyan, Vedaldi y Zisserman calcularon el gradiente del puntaje de la clase respecto de la imagen, y como mapa tomaron el máximo valor absoluto entre los canales de color: exactamente la línea que escribimos. En el mismo paper está la otra técnica de hoy: generar la imagen que maximiza el puntaje de una clase.

> **Adebayo et al., 2018 — Sanity Checks for Saliency Maps** · [arXiv:1810.03292](https://arxiv.org/abs/1810.03292)
>
> **¿depende del modelo?** · «some existing saliency methods are independent both of the model and of the data generating process».

Y este es el control. Adebayo y coautores aleatorizaron los pesos de redes entrenadas y encontraron que algunos métodos de saliencia dan casi el mismo mapa: no dependen ni del modelo ni de los datos. Se parecen más a un detector de bordes que a una explicación. Por eso comparamos con una red al azar. El gradiente simple, el que usamos, pasa esa prueba en el paper; pasarla es necesario, pero no alcanza.

### Oclusión: tapar y medir

Ahora intervenimos. La probabilidad de avión sin tocar nada. Recorremos la imagen con un parche de ocho por ocho píxeles, corriéndolo de a cuatro: cuarenta y nueve posiciones. Ponemos el parche en cero, que después de normalizar es un gris medio, y medimos cuánto baja la probabilidad. Separamos los parches que tocan el avión de los que tapan solo fondo. Sin tapar, cero coma noventa y nueve. Tapando el avión, cae en promedio cero coma veintitrés. Tapando solo fondo, cero coma veinticuatro: lo mismo. Y la mayor caída, cero coma noventa y ocho, sucede tapando fondo.

In [11]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep06_oclusion.py
__file__ = 'snippets/ep06_oclusion.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, sys
import torch
AQUI = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, AQUI)
from _redes import cargar_modelo
from _cifar import cargar, normalizar, CLASES
from _figs import guardar, gris
torch.set_grad_enabled(False)
cnn, _ = cargar_modelo(f"{AQUI}/../datos/cnn_ep05.pt")
X, y = cargar("test", n=100)
i = 21                                             # el mismo avión
avion = gris(X[i].float() / 255) < 0.5             # máscara: píxeles oscuros = el avión

In [12]:
x, k = normalizar(X[i:i + 1]), y[i].item()
p0 = cnn(x).softmax(-1)[0, k].item()
caida = torch.zeros(7, 7)
tapa_avion = torch.zeros(7, 7, dtype=torch.bool)
for a in range(7):
    for b in range(7):                        # parche de 8×8, paso 4
        tapada = x.clone()
        tapada[:, :, 4*a:4*a + 8, 4*b:4*b + 8] = 0     # 0 = gris medio
        caida[a, b] = p0 - cnn(tapada).softmax(-1)[0, k].item()
        tapa_avion[a, b] = avion[4*a:4*a + 8, 4*b:4*b + 8].any()
print(f"p({CLASES[k]}) sin tapar: {p0:.2f}")
print(f"tapando parte del avión: caída media {caida[tapa_avion].mean():.2f}")
print(f"tapando solo fondo:      caída media {caida[~tapa_avion].mean():.2f}")
print(f"la mayor caída: {caida.max():.2f}, en",
      "el avión" if tapa_avion.flatten()[caida.argmax()] else "el fondo")

p(avión) sin tapar: 0.99
tapando parte del avión: caída media 0.23
tapando solo fondo:      caída media 0.24
la mayor caída: 0.98, en el fondo


In [13]:
# Verificación extra (corrió junto con el video)
guardar("ep06_ocl.json", {"caida": [[round(v, 2) for v in fila] for fila in caida.tolist()]})

### Cuánto cae p(avión) al tapar cada zona

El mapa de caídas, siete por siete. Las celdas más fuertes están abajo a la izquierda y arriba a la derecha: zonas de fondo blanco, alrededor del avión. Un parche gris en el blanco baja la probabilidad de avión casi a cero. Tapar el avión mismo cambia menos. En esta imagen, para esta red, el fondo liso y claro es parte de lo que significa avión, y la saliencia no lo mostraba. Un detalle: un parche gris también es algo que la red nunca vio, así que la oclusión tiene su propio sesgo.

## Parte 4 · La imagen que más activa un canal

En vez de buscar en el dataset, fabricamos la imagen con el mismo gradiente.

### Ascenso por gradiente sobre la imagen

$$
x^{*} = \arg\max_{x}\; a_c(x) - \lambda \lVert x \rVert^2
$$
<small>la activación media del canal c; λ = 0,01 frena los píxeles</small>

$$
x \leftarrow x + \eta\, \nabla_x \big(a_c(x) - \lambda \lVert x \rVert^2\big)
$$
<small>lo mismo que entrenar, pero se mueve x y los pesos quedan quietos</small>

Buscamos la imagen x estrella que hace más grande la activación media de un canal c, a sub c. El término lambda por la norma de x al cuadrado castiga los píxeles enormes; sin él, la solución es subir el contraste hasta el infinito. El método es el de siempre, pero al revés: en vez de mover los pesos para bajar la pérdida, dejamos los pesos quietos y movemos los píxeles en la dirección del gradiente, para subir. Eta es el paso. En el dibujo, cada punto sube por la pendiente.

### De ruido a patrón

Arrancamos de ruido chico. El optimizador recibe la imagen como único parámetro: los pesos de la red están congelados. Pasamos la imagen hasta la salida de la tercera convolución y su ReLU, y tomamos un canal. La pérdida es menos la activación más el castigo: minimizarla es maximizar la activación. Después de doscientos pasos, los canales cero y uno llegan a una activación media de tres coma seis y tres coma nueve; el canal dos, a uno.

In [14]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep06_maximizar.py
__file__ = 'snippets/ep06_maximizar.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, sys
import torch
AQUI = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, AQUI)
from _redes import cargar_modelo
from _figs import norm, mosaico, guardar
cnn, _ = cargar_modelo(f"{AQUI}/../datos/cnn_ep05.pt")
for p in cnn.parameters():
    p.requires_grad_(False)
torch.manual_seed(0)

In [15]:
def maximizar(canal, pasos=200, lr=0.05):
    x = (torch.randn(1, 3, 32, 32) * 0.1).requires_grad_(True)  # ruido
    opt = torch.optim.Adam([x], lr=lr)
    for _ in range(pasos):
        act = cnn.features[:8](x)[0, canal]   # salida de conv3 + ReLU
        perdida = -act.mean() + 0.01 * x.pow(2).sum()   # subir, sin explotar
        opt.zero_grad(); perdida.backward(); opt.step()
    return x.detach()[0], act.mean().item()

for canal in [0, 1, 2]:
    img, a = maximizar(canal)
    print(f"canal {canal}: activación media {a:.2f}")

canal 0: activación media 3.64


canal 1: activación media 3.89
canal 2: activación media 1.00


In [16]:
# Verificación extra (corrió junto con el video)
from _figs import gris
torch.manual_seed(0)
paneles = []
for canal in [0, 1, 2]:
    img, _ = maximizar(canal)
    paneles.append(norm(gris(img)))
guardar("ep06_max.json", {"paneles": [p.tolist() for p in paneles]})

### Lo que más activa tres canales de conv3

Lo que sale no son aviones ni perros: son texturas, franjas repetidas con una orientación. Tiene sentido: cada neurona de la tercera capa ve un pedazo de la imagen, y la activación media del canal sube si el patrón se repite por todos lados. Para obtener imágenes que se parezcan a objetos hace falta más: regularizar distinto, desplazar la imagen en cada paso, y redes más grandes. Con una red chica y treinta y dos píxeles, esto es lo honesto que se puede mostrar.

## Resumiendo

- Los filtros de conv1 son plantillas: bordes de luz y contrastes de color
- Hooks: guardar activaciones; cuanto más adentro, más selectivas
- Saliencia: M = max_c |∂S_k/∂x|, con control de red al azar
- Oclusión, en esta imagen: tapar el fondo pesa igual que tapar el avión
- Ascenso por gradiente sobre x: texturas, no objetos
- Próximo: por qué apilar más capas entrena peor, y ResNet